# **Определение эмоциональной окраски отзывов для сети продуктовых магазинов "Пятерочка"**

## **Загрузка датасета с kaggle**

Загрузим открытый датасет от Яндекса с отзывами (ссылка на kaggle: https://www.kaggle.com/datasets/kyakovlev/yandex-geo-reviews-dataset-2023?select=geo-reviews-dataset-2023.csv)

In [2]:
import pandas as pd

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

path = kagglehub.dataset_download("kyakovlev/yandex-geo-reviews-dataset-2023")
file_path = path + "/geo-reviews-dataset-2023.csv"
df =  pd.read_csv(file_path)

Using Colab cache for faster access to the 'yandex-geo-reviews-dataset-2023' dataset.


In [4]:
df.head(10)

,address,name_ru,rating,rubrics,text
0,"Екатеринбург, ул. Московская / ул. Волгоградск...",Московский квартал,3.0,Жилой комплекс,Московский квартал 2.\nШумно : летом по ночам ...
1,"Московская область, Электросталь, проспект Лен...",Продукты Ермолино,5.0,Магазин продуктов;Продукты глубокой заморозки;...,"Замечательная сеть магазинов в общем, хороший ..."
2,"Краснодар, Прикубанский внутригородской округ,...",LimeFit,1.0,Фитнес-клуб,"Не знаю смутят ли кого-то данные правила, но я..."
3,"Санкт-Петербург, проспект Энгельса, 111, корп. 1",Snow-Express,4.0,Пункт проката;Прокат велосипедов;Сапсёрфинг,Хорошие условия аренды. \nДружелюбный персонал...
4,"Тверь, Волоколамский проспект, 39",Студия Beauty Brow,5.0,"Салон красоты;Визажисты, стилисты;Салон бровей...",Топ мастер Ангелина топ во всех смыслах ) Немн...
5,"Иркутская область, Черемхово, Первомайская ули...",Tele2,5.0,Оператор сотовой связи;Интернет-провайдер,"Приятное общение, все доступно объяснили, мне ..."
6,"Воронежская область, Богучарский район, М-4 До...",У тещи,4.0,Кафе,Глубинка страны во всех своих проявлениях. Асс...
7,"Пермь, улица Солдатова, 15",Smoking Park,5.0,Вейп-шоп;Магазин табака и курительных принадле...,"Лучший шоп на крохалях \nБольшоц ассортимент, ..."
8,"Москва, 4-й Кожевнический переулок, 4",Jinju,5.0,Кафе;Кофейня,"5 из 5🖤 Пил кофе и в Риме, и в Париже, но вку..."
9,"Москва, 4-й Кожевнический переулок, 4",Jinju,4.0,Кафе;Кофейня,"Не очень удобное расположение, от метро идти м..."


In [8]:
# Ищем место с наибольшим количеством отзывов во всем датасете
top_place_overall = df['name_ru'].value_counts().head(1)
print(f"Место с наибольшим количеством отзывов: {top_place_overall.index[0]}")
print(f"Количество отзывов: {top_place_overall.values[0]}")

Место с наибольшим количеством отзывов: Пятёрочка
Количество отзывов: 6030


In [14]:
# Создаем датасет только с отзывами для Пятерочки
pyaterochka_df = df[df['name_ru'].str.contains('Пятёрочка|Пятерочка', na=False, case=False)].copy()

In [15]:
# Смотрим распределение отзывов по городам
pyaterochka_df['city'] = pyaterochka_df['address'].str.extract(r'([А-ЯЁ][а-яё]+)')
print("\nТоп-10 городов по количеству отзывов:")
print(pyaterochka_df['city'].value_counts().head(10))


Топ-10 городов по количеству отзывов:
city
Москва           1050
Московская        790
Республика        367
Санкт             301
Краснодарский     259
Свердловская      167
Ленинградская     166
Ростовская        117
Нижний            111
Воронеж           101
Name: count, dtype: int64


## **Анализ тональности отзывов**

Подготовка текста:

In [13]:
import re
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("blanchefort/rubert-base-cased-sentiment")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [16]:
def preprocess_text(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text)
    # Удаляем URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Удаляем лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text).strip()
    # Токенизируем и обрезаем до 512 токенов
    tokens = tokenizer.encode(text, truncation=True, max_length=512)
    # Декодируем обратно в текст
    preprocessed_text = tokenizer.decode(tokens, skip_special_tokens=True)

    return preprocessed_text

In [17]:
pyaterochka_df['preprocessed_text'] = pyaterochka_df['text'].apply(preprocess_text)
pyaterochka_df.head(10)

,address,name_ru,rating,rubrics,text,city,preprocessed_text
95,"Санкт-Петербург, улица Коллонтай, 12, корп. 1",Пятёрочка,5.0,Супермаркет,"Чистота, без просроченных продуктов, персонал ...",Санкт,"Чистота, без просроченных продуктов, персонал ..."
181,"Саратов, Московское шоссе, 18/1",Пятёрочка,5.0,Супермаркет,Очень хочется отметить добродушный персонал и ...,Саратов,Очень хочется отметить добродушный персонал и ...
183,"Московская область, Мытищи, Олимпийский проспе...",Пятёрочка,5.0,Супермаркет,Оченьчистый и просторный магазин. Персонал веж...,Московская,Оченьчистый и просторный магазин. Персонал веж...
395,"Московская область, Богородский городской окру...",Пятёрочка,4.0,Супермаркет,"Большой ассортимент, чисто, большой выбор, всё...",Московская,"Большой ассортимент, чисто, большой выбор, всё..."
425,"Рязань, район Песочня, улица Новосёлов, 40А",Пятёрочка,5.0,Супермаркет,Магазин после новогодних праздников обновился....,Рязань,Магазин после новогодних праздников обновился....
572,"Краснодарский край, муниципальное образование ...",Пятёрочка,5.0,Супермаркет,"Чисто, современно, можно сварить кофе в кофема...",Краснодарский,"Чисто, современно, можно сварить кофе в кофема..."
627,"Орёл, улица Металлургов, 15",Пятёрочка,4.0,Супермаркет,Хороший магазин еды много а так магазин как ма...,Орёл,Хороший магазин еды много а так магазин как ма...
833,"Санкт-Петербург, Парашютная улица, 10",Пятёрочка,4.0,Супермаркет,"Персонал - обычный, ассортимент не большой, но...",Санкт,"Персонал - обычный, ассортимент не большой, но..."
834,"Санкт-Петербург, Парашютная улица, 10",Пятёрочка,1.0,Супермаркет,"Персонал старается, выбор товаров есть, но не ...",Санкт,"Персонал старается, выбор товаров есть, но не ..."
1034,"Московская область, Раменский городской округ,...",Пятёрочка,5.0,Супермаркет,Хороший магазин в шаговой доступности. Всё ест...,Московская,Хороший магазин в шаговой доступности. Всё ест...


Загрузка модели и определение тональности отзывов:

In [19]:
from transformers import pipeline

#загружаем модель
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="blanchefort/rubert-base-cased-sentiment"
)

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
def analyze_text(text):
    if pd.isna(text) or text == "":
        return 0, 'neutral'

    analyze_result = sentiment_pipeline(text)[0]

    label = analyze_result['label']
    score = analyze_result['score']

    if label == 'POSITIVE':
        return score, 'positive'
    elif label == 'NEGATIVE':
        return -score, 'negative'
    else:
        return 0, 'neutral'


Получим оценку тональности каждого отзыва на всем датасете Пятерочки:

In [28]:
from tqdm import tqdm
tqdm.pandas(desc="Анализ тональности")

def progress_bar_anlyze(text):
    return analyze_text(text)

# Применяем ко всем отзывам
results = pyaterochka_df['preprocessed_text'].progress_apply(progress_bar_anlyze)

# Извлекаем результаты
scores = [r[0] for r in results]
sentiments = [r[1] for r in results]

# Записываем в датафрейм
pyaterochka_df['sentiment_score'] = scores
pyaterochka_df['sentiment'] = sentiments

Анализ тональности: 100%|██████████| 6042/6042 [22:24<00:00,  4.49it/s]


Смотрим на общее распрделение тональности:

In [29]:
print(pyaterochka_df['sentiment'].value_counts(normalize=True).round(3) * 100)

sentiment
positive    78.2
negative    12.9
neutral      8.9
Name: proportion, dtype: float64


**Вывод:** 78% положительных отзывов - это хороший показатель, большинство покупателей довольны. 13% негативных отзывов говорит о наличии проблем, поскольку сеть "Пятерочка" - одна из самых популярных продуктовых сетей в России и имеет тысячи магазинов, то 13% - достаточно высокий показатель: каждый 8 отзыв негативный (1/0.129). Было бы хорошо снизить долю негативных отзывов.

## **Сохранение результатов для последующего анализа**

In [30]:
# Сохраняем полный датасет с тональностью
final_file = 'pyaterochka_reviews_with_sentiment.csv'
pyaterochka_df.to_csv(final_file, index=False)

In [31]:
# Скачиваем файл
from google.colab import files
files.download(final_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>